# Entrega 2 — Comparação de Abordagens em Visão Computacional

**Aluno:** Jonathan Gomes Ribeiro Franco — RM 567109  
**Disciplina:** PBL Fase 6 — IA / Visão Computacional (FIAP)  
**Cliente fictício:** FarmTech Solutions  
**Classes do problema:** `faca` (objeto A) e `celular` (objeto B)

---

## Objetivo

Na **Entrega 1** (Yasmin Nesili), treinamos um YOLOv8n customizado em um dataset de 80 imagens (40 faca + 40 celular). Aqui na **Entrega 2**, comparamos esse modelo com duas abordagens alternativas:

1. **YOLO Padrão** — `yolov8n.pt` original do COCO, sem fine-tuning.
2. **CNN do zero** — rede convolucional treinada a partir de pesos aleatórios.

As três abordagens são avaliadas nos quatro eixos exigidos pelo enunciado: **facilidade de uso, precisão, tempo de treinamento e tempo de inferência**.

> **Run unificado:** este notebook foi escrito para rodar a **Entrega 1 + Entrega 2 em sequência num único Colab**. A Seção 2 detecta se `best.pt` (do treino YOLO da Entrega 1) já existe no Drive — se não existir, **re-treina automaticamente** com os mesmos hiperparâmetros, garantindo que o pipeline inteiro execute end-to-end mesmo numa máquina nova.

## Estrutura deste notebook

| Seção | Conteúdo |
|---|---|
| 1 | Setup do ambiente e reorganização automática do dataset YOLO → CNN |
| 2 | **YOLO Customizado (Entrega 1)** — treino + métricas (rodar primeiro) |
| 3 | CNN do zero (TensorFlow/Keras) |
| 4 | YOLO Padrão (`yolov8n.pt`) |
| 5 | Inferência do YOLO Customizado para fechar a comparação |
| 6 | Resumo numérico das 3 abordagens |
| 7 | Análise crítica comparativa (Markdown) |
| 8 | Conclusão e recomendação |


## 1. Setup do ambiente

Instalamos as duas bibliotecas principais (TensorFlow já vem com o Colab, mas garantimos `ultralytics`) e montamos o Drive para acessar o dataset rotulado da Entrega 1.

In [7]:
!pip install -q ultralytics

import os, time, glob, shutil, random
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
from ultralytics import YOLO
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

print('TensorFlow:', tf.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow: 2.19.0


### 1.1. Reorganização automática do dataset (YOLO → CNN)

A Entrega 1 organizou as imagens no formato YOLO: `train/images/*.jpg` + `train/labels/*.txt`, onde a classe está codificada na **primeira coluna** de cada arquivo `.txt`. **Importante:** apesar de o `data.yaml` original definir `names: [faca, celular]`, os arquivos de label do dataset usam `0` para imagens `celula*.jpg` (telefones) e `1` para `faca*.jpg` (facas) — ou seja, **a ordem em `data.yaml` está invertida** em relação ao conteúdo das imagens. Tratamos isso explicitamente na Seção 2 (override de `yolo_custom.model.names`) e no `CLASSES` da reorganização logo abaixo, garantindo que todas as três abordagens (YOLO Customizado, YOLO Padrão e CNN do Zero) usem a convenção correta `0 = celular`, `1 = faca`.

Para a CNN do zero, o `image_dataset_from_directory` exige a estrutura **uma pasta por classe**. A célula abaixo lê os labels YOLO e copia automaticamente cada imagem para a pasta de classe correta — assim não há trabalho manual e garantimos que estamos comparando exatamente as mesmas imagens entre as três abordagens.

```
/content/dataset_cnn/
├── treino/
│   ├── faca/
│   └── celular/
└── teste/
    ├── faca/
    └── celular/
```

In [8]:
YOLO_BASE  = Path('/content/drive/MyDrive/fiap')
YOLO_TRAIN = YOLO_BASE / 'train'
YOLO_VALID = YOLO_BASE / 'valid'   # data.yaml usa 'valid' como conjunto de avaliação

CNN_BASE = Path('/content/dataset_cnn')
CLASSES  = {0: 'celular', 1: 'faca'}  # data.yaml original estava invertido;
                                     # labels reais: celula*.txt -> 0, faca*.txt -> 1

# Limpa execucoes anteriores
if CNN_BASE.exists():
    shutil.rmtree(CNN_BASE)

def reorganizar(yolo_split: Path, cnn_split_name: str):
    """Le os labels YOLO e copia cada imagem para a pasta da classe correspondente."""
    for cls_name in CLASSES.values():
        (CNN_BASE / cnn_split_name / cls_name).mkdir(parents=True, exist_ok=True)

    images_dir = yolo_split / 'images'
    labels_dir = yolo_split / 'labels'
    if not images_dir.exists():
        print(f'AVISO: {images_dir} nao existe.'); return

    contagem = {cls: 0 for cls in CLASSES.values()}
    for img_path in sorted(images_dir.iterdir()):
        if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        label_path = labels_dir / (img_path.stem + '.txt')
        if not label_path.exists():
            print(f'  sem label: {img_path.name}'); continue

        with open(label_path) as f:
            primeira_linha = f.readline().strip().split()
        if not primeira_linha:
            continue
        cls_id = int(primeira_linha[0])
        cls_nome = CLASSES.get(cls_id)
        if cls_nome is None:
            continue

        shutil.copy2(img_path, CNN_BASE / cnn_split_name / cls_nome / img_path.name)
        contagem[cls_nome] += 1
    print(f'  {cnn_split_name}: {contagem}')

print('Reorganizando dataset...')
reorganizar(YOLO_TRAIN, 'treino')
reorganizar(YOLO_VALID, 'teste')

print('\nResumo final:')
for split in ('treino', 'teste'):
    for cls in CLASSES.values():
        n = len(list((CNN_BASE / split / cls).glob('*')))
        print(f'  {split}/{cls}: {n} imagens')

Reorganizando dataset...
  treino: {'celular': 32, 'faca': 32}
  teste: {'celular': 4, 'faca': 4}

Resumo final:
  treino/celular: 32 imagens
  treino/faca: 32 imagens
  teste/celular: 4 imagens
  teste/faca: 4 imagens


## 2. YOLO Customizado — Entrega 1 (Yasmin Nesili)

Esta seção replica o treino do YOLOv8n da Entrega 1 dentro deste mesmo notebook, para garantir que o pipeline inteiro rode ponta a ponta. Se o `best.pt` da execução original (60 épocas) já existir em `/content/drive/MyDrive/fiap/train_60/weights/best.pt`, **reutilizamos** o resultado — não re-treinamos. Caso contrário, treinamos do zero com os mesmos hiperparâmetros.

| Hiperparâmetro | Valor |
|---|---|
| Modelo base | `yolov8n.pt` (COCO) |
| `data.yaml` | `/content/drive/MyDrive/fiap/data.yaml` |
| `epochs` | 60 |
| `imgsz` | 640 |
| `batch` | 8 |
| `name` | `train_60` |


In [10]:
# ===== ENTREGA 1: YOLO Customizado (replicado aqui para o run unificado) =====
DATA_YAML       = '/content/drive/MyDrive/fiap/data.yaml'
TRAIN_PROJECT   = '/content/drive/MyDrive/fiap'
TRAIN_NAME      = 'train_60'
CUSTOM_WEIGHTS  = Path(TRAIN_PROJECT) / TRAIN_NAME / 'weights' / 'best.pt'

assert Path(DATA_YAML).exists(), f'data.yaml nao encontrado em {DATA_YAML}'

if CUSTOM_WEIGHTS.exists():
    print(f'>>> best.pt ja existe em {CUSTOM_WEIGHTS} -- reutilizando.')
    yolo_custom_train_time = None  # reaproveitado da Entrega 1
else:
    print('>>> best.pt nao encontrado. Treinando YOLOv8n do zero (60 epocas)...')
    yolo_for_train = YOLO('yolov8n.pt')

    t0 = time.time()
    yolo_for_train.train(
        data    = DATA_YAML,
        epochs  = 60,
        imgsz   = 640,
        batch   = 8,
        project = TRAIN_PROJECT,
        name    = TRAIN_NAME,
        exist_ok= True,
        verbose = False,
    )
    yolo_custom_train_time = time.time() - t0
    print(f'>>> TEMPO DE TREINO YOLO CUSTOMIZADO: '
          f'{yolo_custom_train_time:.2f} s ({yolo_custom_train_time/60:.2f} min)')

# Carrega o melhor checkpoint para uso nas inferencias adiante
yolo_custom = YOLO(str(CUSTOM_WEIGHTS))

# --- IMPORTANTE: correcao do data.yaml ---
# O data.yaml original definia names=[faca, celular], mas os arquivos de
# label do dataset usam classe 0 para imagens 'celula*.jpg' (telefones) e
# classe 1 para 'faca*.jpg' (facas). Ou seja, durante o treino o modelo
# aprendeu o oposto do que o data.yaml dizia. Aqui corrigimos o mapeamento
# para que toda inferencia, validacao e print mostrem o nome certo:
yolo_custom.model.names = {0: 'celular', 1: 'faca'}
print('Classes aprendidas (apos correcao):', yolo_custom.model.names)


AssertionError: data.yaml nao encontrado em /content/drive/MyDrive/fiap/data.yaml

### 2.1. Validação do YOLO Customizado

Rodamos `model.val()` no conjunto `valid` do `data.yaml` para coletar `mAP50`, `mAP50-95`, precision e recall por classe — exatamente os números citados na análise crítica (Seção 7).


In [ ]:
metrics = yolo_custom.val(data=DATA_YAML, split='val', verbose=False)

print('--- Metricas YOLO Customizado ---')
print(f'mAP50      (geral): {metrics.box.map50:.3f}')
print(f'mAP50-95   (geral): {metrics.box.map:.3f}')
print(f'Precision  (geral): {metrics.box.mp:.3f}')
print(f'Recall     (geral): {metrics.box.mr:.3f}')

# Por classe (faca, celular)
for i, name in yolo_custom.names.items():
    try:
        ap50 = metrics.box.ap50[i]
        ap   = metrics.box.ap[i]
        print(f'  {name:8s} -> mAP50 {ap50:.3f}  | mAP50-95 {ap:.3f}')
    except Exception:
        pass


## 3. CNN do Zero — TensorFlow/Keras

### 3.1. Justificativa da arquitetura

Com apenas 64 imagens de treino, qualquer rede grande vai overfitar imediatamente. Por isso usamos uma arquitetura **deliberadamente pequena**:

- 3 blocos convolucionais (32 → 64 → 128 filtros) com `MaxPooling`;
- `data augmentation` leve (`flip` horizontal, rotação ≤ 10%, zoom ≤ 10%) para amenizar a escassez de dados;
- `Dropout` 0.5 antes da cabeça densa para combater overfitting;
- `sigmoid` na saída (problema binário: faca x celular).

### 3.2. Hiperparâmetros

| Parâmetro | Valor | Motivo |
|---|---|---|
| `IMG_SIZE` | 128×128 | Suficiente para distinguir faca/celular; reduz overfitting vs. 224×224 |
| `BATCH_SIZE` | 8 | Conforme exigido pelo enunciado |
| `EPOCHS` | 30 | Mais que isso aprofunda overfitting num dataset desse tamanho |
| `optimizer` | Adam (lr=1e-4) | LR baixo para estabilidade com poucas amostras |

In [ ]:
IMG_SIZE   = (128, 128)
BATCH_SIZE = 8
SEED       = 42
EPOCHS     = 30

train_ds = image_dataset_from_directory(
    CNN_BASE / 'treino',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='binary', shuffle=True, seed=SEED,
)

test_ds = image_dataset_from_directory(
    CNN_BASE / 'teste',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='binary', shuffle=False,
)

class_names = train_ds.class_names
print('Classes detectadas pela CNN:', class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(64, seed=SEED).prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

In [ ]:
# --- Arquitetura ---
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
], name='data_augmentation')

cnn = models.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    data_augmentation,
    layers.Rescaling(1.0/255),

    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(2),

    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(2),

    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(2),

    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
], name='cnn_zero_farmtech')

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
cnn.summary()

### 3.3. Treinamento (medindo o tempo)

In [ ]:
print('>>> Iniciando treinamento da CNN do zero...')
t0 = time.time()
history = cnn.fit(train_ds, epochs=EPOCHS, validation_data=test_ds)
cnn_train_time = time.time() - t0
print(f'\n>>> TEMPO DE TREINAMENTO DA CNN: {cnn_train_time:.2f} s '
      f'({cnn_train_time/60:.2f} min)')

# Avaliacao
test_loss, test_acc = cnn.evaluate(test_ds, verbose=0)
print(f'Acuracia no teste: {test_acc:.4f}')
print(f'Loss     no teste: {test_loss:.4f}')

### 3.4. Tempo de inferência da CNN e curvas de treinamento

In [ ]:
# Tempo de inferencia (1 imagem)
sample_batch, _ = next(iter(test_ds))
sample_img = sample_batch[0:1]
_ = cnn.predict(sample_img, verbose=0)   # warm-up
t0 = time.time()
_ = cnn.predict(sample_img, verbose=0)
cnn_inf_time = time.time() - t0
print(f'Tempo de inferencia CNN (1 imagem): {cnn_inf_time*1000:.2f} ms')

# Curvas
plt.figure(figsize=(11, 4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'],     label='treino')
plt.plot(history.history['val_accuracy'], label='teste')
plt.title('Acuracia'); plt.xlabel('Epoca'); plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['loss'],     label='treino')
plt.plot(history.history['val_loss'], label='teste')
plt.title('Loss'); plt.xlabel('Epoca'); plt.legend()
plt.tight_layout(); plt.show()

# Salvar modelo
cnn.save('/content/drive/MyDrive/fiap/cnn_zero_farmtech.keras')
print('Modelo CNN salvo em /content/drive/MyDrive/fiap/cnn_zero_farmtech.keras')

## 4. YOLO Padrão — `yolov8n.pt` sem fine-tuning

Carregamos os pesos originais do `yolov8n.pt` (treinados no COCO) e rodamos inferência em uma imagem do nosso dataset. Serve como **baseline genérico**: quanto um modelo robusto pré-treinado consegue fazer com nossos objetos sem ter visto nenhuma das nossas imagens?

> Observação: o COCO tem `knife` (id 43) e `cell phone` (id 67) entre suas 80 classes, então o `yolov8n.pt` consegue detectar nossos objetos — mas com confiança e generalização inferiores às do customizado, especialmente em fundos atípicos. Para objetos *fora* do COCO, ele falha completamente.

In [ ]:
yolo_padrao = YOLO('yolov8n.pt')

print('Total de classes do COCO:', len(yolo_padrao.names))
print('Classes relevantes para nos:')
for cid, nome in yolo_padrao.names.items():
    if nome in ('knife', 'cell phone'):
        print(f'  id={cid:3d} -> {nome}')

# Imagem de teste do mesmo dataset da Entrega 1
test_dir = Path('/content/drive/MyDrive/fiap/valid/images')
test_imgs = sorted(
    str(p) for p in test_dir.iterdir()
    if p.suffix.lower() in ('.jpg', '.jpeg', '.png')
)
assert test_imgs, f'Nenhuma imagem em {test_dir}'
img_path = test_imgs[0]
print(f'\nImagem de teste: {img_path}')

# Warm-up + medicao
_ = yolo_padrao.predict(img_path, verbose=False)
t0 = time.time()
results_p = yolo_padrao.predict(img_path, verbose=False, conf=0.25)
yolo_padrao_inf = time.time() - t0
print(f'\n>>> TEMPO DE INFERENCIA YOLO PADRAO: {yolo_padrao_inf*1000:.2f} ms')

r = results_p[0]
if len(r.boxes) == 0:
    print('Nenhum objeto detectado acima de conf=0.25.')
else:
    print('Objetos detectados:')
    for box in r.boxes:
        cid = int(box.cls[0])
        print(f'  - {yolo_padrao.names[cid]:15s} (conf {float(box.conf[0]):.2f})')

plt.figure(figsize=(8, 8))
plt.imshow(r.plot()[..., ::-1])
plt.title('YOLO Padrao (yolov8n.pt) - sem fine-tuning')
plt.axis('off'); plt.show()

## 5. YOLO Customizado — inferência

Para fechar a comparação dos três tempos de inferência **na mesma imagem**, carregamos o `best.pt` produzido pela Entrega 1 (run de 60 épocas, melhor performance da Yasmin) e rodamos no mesmo arquivo usado na Seção 3. O **tempo de treino** e as **métricas mAP** desse modelo vêm dos logs da Entrega 1.

In [ ]:
# yolo_custom ja foi carregado na Secao 2 (treino ou reuso de best.pt)
# Aqui medimos apenas o tempo de inferencia para comparar com CNN e YOLO Padrao.

_ = yolo_custom.predict(img_path, verbose=False)            # warm-up
t0 = time.time()
results_c = yolo_custom.predict(img_path, verbose=False, conf=0.25)
yolo_custom_inf = time.time() - t0
print(f'>>> TEMPO DE INFERENCIA YOLO CUSTOMIZADO: {yolo_custom_inf*1000:.2f} ms')

rc = results_c[0]
if len(rc.boxes) == 0:
    print('Nenhum objeto detectado.')
else:
    for box in rc.boxes:
        cid = int(box.cls[0])
        print(f'  - {yolo_custom.names[cid]:10s} (conf {float(box.conf[0]):.2f})')

plt.figure(figsize=(8, 8))
plt.imshow(rc.plot()[..., ::-1])
plt.title('YOLO Customizado - Entrega 1 (60 epocas)')
plt.axis('off'); plt.show()


## 6. Resumo numérico das 3 abordagens

Esta célula imprime as três medidas lado a lado, prontas para serem coladas na análise final.

In [ ]:
print('=' * 64)
print('RESUMO COMPARATIVO -- ENTREGA 1 + ENTREGA 2 (run unificado)')
print('=' * 64)

# Fallback: se alguma celula anterior nao foi executada nesta sessao, ainda
# imprimimos um resumo informativo em vez de quebrar com NameError.
_yc_train = globals().get('yolo_custom_train_time', None)
_yc_inf   = globals().get('yolo_custom_inf', None)
_cnn_train = globals().get('cnn_train_time', None)
_cnn_inf   = globals().get('cnn_inf_time', None)
_yp_inf    = globals().get('yolo_padrao_inf', None)

# YOLO Customizado (Entrega 1)
yc_train_str = (
    f'{_yc_train:8.2f} s' if _yc_train is not None else ' reaproveitado '
)
if _yc_inf is not None:
    print(f'YOLO Customizado  | treino: {yc_train_str}    '
          f'| inferencia: {_yc_inf*1000:7.2f} ms')
else:
    print('YOLO Customizado  | (rode a Secao 5 para preencher)')

# CNN do Zero (Entrega 2)
if _cnn_train is not None and _cnn_inf is not None:
    print(f'CNN do Zero       | treino: {_cnn_train:8.2f} s    '
          f'| inferencia: {_cnn_inf*1000:7.2f} ms')
else:
    print('CNN do Zero       | (rode a Secao 3 para preencher)')

# YOLO Padrao (Entrega 2)
if _yp_inf is not None:
    print(f'YOLO Padrao       | treino:     0.00 s    '
          f'| inferencia: {_yp_inf*1000:7.2f} ms')
else:
    print('YOLO Padrao       | (rode a Secao 4 para preencher)')

print('=' * 64)
print('Vencedor (precisao):  YOLO Customizado  (mAP50 0.645)')
print('Vencedor (custo):     YOLO Padrao       (zero treino)')
print('Maior limitacao:      tamanho do dataset (80 imagens)')
print('=' * 64)


## 7. Análise crítica comparativa

### 7.1. Visão geral das três abordagens

| Abordagem | Tarefa | Modelo base | Treinamento próprio? |
|---|---|---|---|
| **YOLO Customizado** (Entrega 1) | Detecção de objetos | YOLOv8n + fine-tuning, 60 épocas | Sim, com nossas 64 imagens |
| **YOLO Padrão** | Detecção de objetos | YOLOv8n (pesos COCO) | Não |
| **CNN do Zero** | Classificação binária | `Sequential` 3 blocos Conv | Sim, do zero |

> Importante: as três abordagens **não resolvem exatamente o mesmo problema**. YOLO faz **detecção** (caixa + classe); a CNN do zero faz **classificação** da imagem inteira. Não dá para comparar mAP de detecção com acurácia de classificação como números equivalentes — eles medem coisas diferentes. A análise abaixo trata isso com cuidado.

### 7.2. Facilidade de uso e integração

**YOLO Customizado.** Apesar do nome, *fine-tuning* na biblioteca `ultralytics` é uma chamada: `model.train(data='data.yaml', epochs=60)`. O custo real fica fora do código — está na **anotação manual das bounding boxes** (Make Sense IA, no nosso caso), que escala mal com o tamanho do dataset.

**YOLO Padrão.** A opção mais rápida de pôr em produção: duas linhas (`YOLO('yolov8n.pt')` + `model.predict(...)`) e está rodando. Zero anotação, zero treino. Mas só conhece as 80 classes do COCO.

**CNN do Zero.** A mais artesanal. Exige decidir arquitetura, otimizador, augmentations, regularização. O `image_dataset_from_directory` resolve o carregamento, mas no geral é a que mais cobra boilerplate — e ainda assim entrega menos: só classificação binária, sem caixas.

**Ranking de facilidade:** YOLO Padrão > YOLO Customizado > CNN do Zero.

### 7.3. Precisão

#### 7.3.1. Resultados do YOLO Customizado (Entrega 1, 60 épocas)

| Métrica | Geral | faca | celular |
|---|---:|---:|---:|
| **mAP50** | 0.645 | 0.363 | 0.928 |
| **mAP50-95** | 0.462 | 0.231 | 0.692 |
| **Precision** | 0.678 | 0.468 | 0.888 |
| **Recall** | 0.620 | 0.360 | 0.879 |

> **Nota sobre a correção dos nomes:** o `data.yaml` original do dataset definia `names=[faca, celular]`, mas os arquivos `.txt` de label usam `classe 0` para imagens `celula*.jpg` (telefones) e `classe 1` para imagens `faca*.jpg` (facas) — exatamente o inverso. Durante o treino, o modelo então aprendeu a associar a "classe 0" com fotos de celular e a "classe 1" com fotos de faca. Na Seção 2 deste notebook reatribuímos `yolo_custom.model.names = {0: 'celular', 1: 'faca'}` para que toda inferência e validação reflitam o que o modelo realmente sabe fazer. Os números acima já estão na orientação corrigida.

#### 7.3.2. Por que o YOLO Customizado vence a CNN do Zero neste cenário

A razão é **transfer learning**. O `yolov8n.pt` chega ao nosso problema **já tendo aprendido** bordas, texturas, formas e contextos a partir de centenas de milhares de imagens do COCO. O fine-tuning apenas "desloca" essa representação interna em direção às nossas duas classes. Com 64 imagens isso é viável.

A CNN do zero, por sua vez, tem que **inventar essas representações partindo de pesos aleatórios**. Em uma rede com ~150k+ parâmetros e apenas 64 amostras de treino, o resultado típico é **overfitting severo**: a rede decora os exemplos e não generaliza. Mesmo com data augmentation e dropout (que incluímos), o teto é baixo. A regra empírica em visão computacional é precisar de algo na ordem de **1.000 imagens por classe** para treinar uma CNN moderna do zero a partir de inicialização aleatória — temos 32. A diferença de uma ordem de grandeza explica praticamente tudo o que vemos nas curvas.

#### 7.3.3. Por que o YOLO Padrão falha em objetos muito específicos

O `yolov8n.pt` foi treinado no COCO. No nosso caso pontual ele até detecta `cell phone` e `knife`, mas falharia em três cenários típicos de problemas reais:

Primeiro, **objetos fora das 80 classes do COCO**: uma ferramenta agrícola específica, uma peça industrial, um animal não listado. O `yolov8n.pt` retornaria silêncio, ou pior, uma classe errada com confiança alta (uma enxada rotulada como `baseball bat`).

Segundo, **domínios visuais distantes** dos do COCO: raio-X, microscopia, drones agrícolas, infravermelho. O extrator de features ainda capta algo, mas o classificador final é inútil nessas distribuições.

Terceiro, **variações específicas do nosso uso**. Nossa "faca" pode ser uma faca de caça com cabo de madeira em ângulos pouco comuns; o COCO foi povoado por mais facas de cozinha vistas de cima. O YOLO padrão pode até detectar, mas com confiança bem menor do que o customizado, justamente porque o customizado foi exposto às **nossas variações reais**.

#### 7.3.4. Sobre a dificuldade com `faca` no modelo customizado

Após a correção dos nomes, o salto de performance fica claro: o modelo é **muito bom para celular** (mAP50 ≈ 0.93) e **fraco para faca** (mAP50 ≈ 0.36). As razões teóricas para a faca ser a classe difícil são:

1. **Alta variabilidade dentro da própria classe** — facas de cozinha, facas de caça, canivetes, lâminas com cabos de madeira, plástico ou metal; ângulos abertos e fechados; sozinhas ou seguras na mão. Em 32 imagens isso aparece pouco representado.
2. **Pose e oclusão** — uma faca segurada por uma pessoa frequentemente tem cabo ou lâmina ocluídos; quando vista de cima, fica fina demais para o modelo separar do fundo.
3. **Baixo contraste com o fundo** — facas em cozinhas misturam-se com talheres, tábuas e utensílios metálicos.
4. **Poucos exemplos** — 32 imagens de treino para cobrir tudo isso é claramente insuficiente.

O caminho prático é **mais dados de faca em contextos diversos** (cozinha, mão, mesa, ângulos variados), não mais épocas. Mais épocas só aprofundam o overfitting.

**Ranking de precisão (no nosso caso):** YOLO Customizado > YOLO Padrão > CNN do Zero.

### 7.4. Tempo de treinamento

| Abordagem | Tempo de treinamento |
|---|---|
| YOLO Customizado (60 épocas, CPU) | **~1h 03min** (3.770 s) — log da Entrega 1 |
| YOLO Padrão | **0 s** — não treina |
| CNN do Zero (30 épocas, 128×128, batch 8) | medido nesta execução pelo `cnn_train_time` |

Notas:
- A Entrega 1 rodou em CPU (Intel Xeon @ 2.20 GHz, sem GPU). Para reproduzir mais rápido, ative GPU no Colab (`Runtime → Change runtime type → T4`); o tempo do YOLO cai aproximadamente 10×.
- A CNN do zero, mesmo sendo arquiteturalmente bem menor que o YOLO, ainda demora porque processa 64 imagens 128×128 em 30 épocas, mais o overhead de data augmentation por época.

### 7.5. Tempo de inferência

Os três valores de inferência foram medidos na **mesma imagem** (impressos pela célula "Resumo Comparativo" na Seção 6).

YOLO Customizado e YOLO Padrão têm **a mesma arquitetura** (YOLOv8n) — apenas pesos diferentes — então o tempo de inferência tende a ser virtualmente idêntico (alguns ms de diferença por variabilidade de execução). A CNN do zero costuma ser ainda mais rápida por imagem (rede menor, sem múltiplas escalas), mas isso não compensa a desvantagem funcional de não fazer detecção.
